In [ ]:
%pip install -q openai tenacity tqdm pandas

In [ ]:
import os
from getpass import getpass

DATA_PATH = "airline_reviews.csv"
REVIEW_COLUMN = "Review"

SAMPLE_SIZE = 20
PROVIDER = "groq"

if PROVIDER == "groq":
    MODEL_NAME = "llama-3.3-70b-versatile"
    BASE_URL = "https://api.groq.com/openai/v1"
    API_KEY = os.environ.get("GROQ_API_KEY") or getpass("Enter your Groq API key: ")
else:
    MODEL_NAME = "gpt-4o-mini"
    BASE_URL = None
    API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")


Enter your Groq API key: ··········


In [ ]:
import pandas as pd

pdf = pd.read_csv('Airline_review.csv')
print(f"Total rows: {len(pdf)}")
pdf.head()


Total rows: 23171


,Unnamed: 0,Airline Name,Overall_Rating,Review_Title,Review Date,Verified,Review,Aircraft,Type Of Traveller,Seat Type,Route,Date Flown,Seat Comfort,Cabin Staff Service,Food & Beverages,Ground Service,Inflight Entertainment,Wifi & Connectivity,Value For Money,Recommended
0,0,AB Aviation,9,"""pretty decent airline""",11th November 2019,True,Moroni to Moheli. Turned out to be a pretty ...,NaN,Solo Leisure,Economy Class,Moroni to Moheli,November 2019,4.0,5.0,4.0,4.0,NaN,NaN,3.0,yes
1,1,AB Aviation,1,"""Not a good airline""",25th June 2019,True,Moroni to Anjouan. It is a very small airline...,E120,Solo Leisure,Economy Class,Moroni to Anjouan,June 2019,2.0,2.0,1.0,1.0,NaN,NaN,2.0,no
2,2,AB Aviation,1,"""flight was fortunately short""",25th June 2019,True,Anjouan to Dzaoudzi. A very small airline an...,Embraer E120,Solo Leisure,Economy Class,Anjouan to Dzaoudzi,June 2019,2.0,1.0,1.0,1.0,NaN,NaN,2.0,no
3,3,Adria Airways,1,"""I will never fly again with Adria""",28th September 2019,False,Please do a favor yourself and do not fly wi...,NaN,Solo Leisure,Economy Class,Frankfurt to Pristina,September 2019,1.0,1.0,NaN,1.0,NaN,NaN,1.0,no
4,4,Adria Airways,1,"""it ruined our last days of holidays""",24th September 2019,True,Do not book a flight with this airline! My fr...,NaN,Couple Leisure,Economy Class,Sofia to Amsterdam via Ljubljana,September 2019,1.0,1.0,1.0,1.0,1.0,1.0,1.0,no


In [ ]:
sample_pdf = pdf.sample(n=min(SAMPLE_SIZE, len(pdf)), random_state=42).reset_index(drop=True)
all_reviews = sample_pdf[REVIEW_COLUMN].fillna("").astype(str).tolist()
len(all_reviews)


20

In [ ]:
tools = [
  {
    "type": "function",
    "function": {
        "name": "extracts_intents",
        "parameters": {
          "type": "object",
          "properties": {
            "intents": {
              "type": "array",
              "description": "List of intents identified from the customer review",
              "items": {
                "type": "object",
                "properties": {
                  "intent": {
                    "type": "string",
                    "description": "Description of the identified intent"
                  },
                  "text_summary": {
                    "type": "string",
                    "description": "Summary of the intent"
                  },
                  "sentiment": {
                    "type": "string",
                    "enum": ["Positive", "Negative", "Neutral"],
                    "description": "Sentiment of the intent"
                  },
                  "named_entities": {
                    "type": "array",
                    "items": {
                      "type": "string",
                      "description": "Named entities in the text, if any, like 'Chicago' or 'XYZ Airlines'"
                    }
                  }
                },
                "required": ["intent", "text_summary", "sentiment"]
              }
            }
          }
        }
    }
  }
]


In [ ]:
import json

example_output = {
  "intents": [
    {
      "intent": "Check-in experience",
      "text_summary": "The check-in process was smooth.",
      "sentiment": "Positive",
      "named_entities": ["XYZ Airlines"]
    },
    {
      "intent": "Seating comfort",
      "text_summary": "The seating was cramped.",
      "sentiment": "Negative",
      "named_entities": []
    },
    {
      "intent": "Food quality",
      "text_summary": "The food quality was below average.",
      "sentiment": "Negative",
      "named_entities": []
    },
    {
      "intent": "Flight attendant service",
      "text_summary": "The flight attendants were very polite and helpful.",
      "sentiment": "Positive",
      "named_entities": []
    },
    {
      "intent": "Baggage issue",
      "text_summary": "I had an issue with my baggage, but it was quickly resolved.",
      "sentiment": "Neutral",
      "named_entities": []
    }
  ]
}

format_instructions = {
  "intents": [
    {
      "intent": "",
      "text_summary": "",
      "sentiment": "",
      "named_entities": [""]
    }
  ]
}


In [ ]:
def build_prompt(review: str) -> str:
    return f"""
Follow instructions below and extract intents from a customer review as a json string. DO NOT include any notes or additional information in the output.

### Instructions:
- **Identify each distinct intent** in the review as "intent". The review may contain multiple distinct intents related to different aspects of the customer's experience (e.g., service, seating, food, check-in, baggage handling).
- **Summarize the text** associated with each intent as "text_summary".
- **Classify the sentiment** (Positive, Negative, or Neutral) of each intent as "sentiment".
- **If applicable, extract any "named entities"**, such as the airline name or specific service mentioned.
- **Return a list of intents a JSON string.** Follow the output format and use example ouput below as a reference. Make sure the JSON string is COMPLETE. Do not include additional information.

### Output Format
{json.dumps(format_instructions)}

### Example Review:
"I flew with XYZ Airlines for a 6-hour flight. The check-in process was smooth, but the seating was cramped, and the food quality was below average. The flight attendants were very polite and helpful. I had an issue with my baggage, but it was quickly resolved."

### Example Output (JSON format):
{json.dumps(example_output)}

### Review to analyze:
{review}
"""


In [ ]:
import concurrent.futures
from openai import OpenAI, RateLimitError
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
    retry_if_exception_type,
)
from tqdm.auto import tqdm
from typing import List

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


@retry(
    wait=wait_random_exponential(min=1, max=30),
    stop=stop_after_attempt(3),
    retry=retry_if_exception_type(RateLimitError),
)
def call_chat_model(review: str, temperature: float = 0.1, max_tokens: int = 500, **kwargs):
    """Calls the chat model and returns the response text or tool calls."""
    chat_args = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful analyst for a major airline. You help analyze customer reviews and extract insights.",
            },
            {
                "role": "user",
                "content": build_prompt(review),
            },
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    chat_args.update(kwargs)

    chat_completion = client.chat.completions.create(**chat_args)
    response = chat_completion.choices[0].message

    if response.tool_calls:
        call_args = [c.function.arguments for c in response.tool_calls]
        if len(call_args) == 1:
            return call_args[0]
        return call_args
    return response.content


def call_in_parallel(func, items: List[str], max_workers: int = 5) -> List:
    """Calls func(item) for all items in parallel and returns responses."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = []
        for r in tqdm(executor.map(func, items), total=len(items)):
            results.append(r)
        return results


In [ ]:
def extract_batch(review: str):
    return call_chat_model(review=review, tools=tools)


results = call_in_parallel(extract_batch, all_reviews, max_workers=1)

results_df_preview = pd.DataFrame({"Review": all_reviews, "Model response": results})
results_df_preview.head()

  0%|          | 0/20 [00:00<?, ?it/s]

,Review,Model response
0,Tel Aviv to Amman. We had a short flight to ...,"{""intents"":[{""intent"":""Flight route and connec..."
1,Flight got delayed when I was flying to DR f...,"{""intents"":[{""intent"":""Flight delay"",""named_en..."
2,"Flying alone, my 14 year old daughter’s fligh...","{""intents"":[{""intent"":""Flight cancellation"",""n..."
3,Pune to Delhi. This airline misleads it’s cu...,"{""intents"":[{""intent"":""Misleading advertising""..."
4,I fly this route from Malta to Amsterdam almo...,"{""intents"":[{""intent"":""Air Malta avoidance"",""n..."


In [ ]:
parsed_data = []
for index, item in enumerate(results):
    try:
        parsed_data.append({"id": index, **json.loads(item)})
    except (json.JSONDecodeError, TypeError):
        parsed_data.append({"id": index, "intents": []})

exploded_df = pd.DataFrame(parsed_data).explode("intents")
exploded_df = exploded_df[exploded_df["intents"].apply(lambda x: isinstance(x, dict))]

results_df = pd.json_normalize(exploded_df["intents"])
results_df["id"] = exploded_df["id"].values
results_df["llm_response"] = exploded_df["intents"].values

results_df.head()


,intent,named_entities,sentiment,text_summary,id,llm_response
0,Flight route and connections,[],Neutral,"Tel Aviv to Amman, connecting to Heathrow and ...",0,"{'intent': 'Flight route and connections', 'na..."
1,In-flight smoking issue,[],Negative,There was smoking on the plane,0,"{'intent': 'In-flight smoking issue', 'named_e..."
2,Flight attendant service,[],Negative,The stewardess spent the entire time on her ce...,0,"{'intent': 'Flight attendant service', 'named_..."
3,Flight delay,[Frontier],Negative,Flight got delayed for an hour,1,"{'intent': 'Flight delay', 'named_entities': [..."
4,Flight schedule change,[Frontier],Neutral,Flight got advanced by 30 minutes,1,"{'intent': 'Flight schedule change', 'named_en..."


In [ ]:
merged_df = pd.merge(sample_pdf, results_df, left_on=sample_pdf.index, right_on="id", how="right")
merged_df.head()


,Unnamed: 0,Airline Name,Overall_Rating,Review_Title,Review Date,Verified,Review,Aircraft,Type Of Traveller,Seat Type,...,Inflight Entertainment,Wifi & Connectivity,Value For Money,Recommended,intent,named_entities,sentiment,text_summary,id,llm_response
0,17155,Royal Jordanian Airlines,2,"""there was smoking on the plane""",27th June 2019,True,Tel Aviv to Amman. We had a short flight to ...,NaN,Couple Leisure,Business Class,...,NaN,NaN,1.0,no,Flight route and connections,[],Neutral,"Tel Aviv to Amman, connecting to Heathrow and ...",0,"{'intent': 'Flight route and connections', 'na..."
1,17155,Royal Jordanian Airlines,2,"""there was smoking on the plane""",27th June 2019,True,Tel Aviv to Amman. We had a short flight to ...,NaN,Couple Leisure,Business Class,...,NaN,NaN,1.0,no,In-flight smoking issue,[],Negative,There was smoking on the plane,0,"{'intent': 'In-flight smoking issue', 'named_e..."
2,17155,Royal Jordanian Airlines,2,"""there was smoking on the plane""",27th June 2019,True,Tel Aviv to Amman. We had a short flight to ...,NaN,Couple Leisure,Business Class,...,NaN,NaN,1.0,no,Flight attendant service,[],Negative,The stewardess spent the entire time on her ce...,0,"{'intent': 'Flight attendant service', 'named_..."
3,10119,Frontier Airlines,1,"""Never flying with them again""",16th July 2023,False,Flight got delayed when I was flying to DR f...,NaN,Solo Leisure,Economy Class,...,1.0,1.0,1.0,no,Flight delay,[Frontier],Negative,Flight got delayed for an hour,1,"{'intent': 'Flight delay', 'named_entities': [..."
4,10119,Frontier Airlines,1,"""Never flying with them again""",16th July 2023,False,Flight got delayed when I was flying to DR f...,NaN,Solo Leisure,Economy Class,...,1.0,1.0,1.0,no,Flight schedule change,[Frontier],Neutral,Flight got advanced by 30 minutes,1,"{'intent': 'Flight schedule change', 'named_en..."


In [ ]:
output_columns = [
    "id", REVIEW_COLUMN, "llm_response", "intent", "text_summary", "sentiment", "named_entities"
]
existing_columns = [c for c in output_columns if c in merged_df.columns]
output_df = merged_df[existing_columns]

OUTPUT_PATH = "raw_intents_output.csv"
output_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(output_df)} rows to {OUTPUT_PATH}")
output_df.head()


Saved 101 rows to raw_intents_output.csv


,id,Review,llm_response,intent,text_summary,sentiment,named_entities
0,0,Tel Aviv to Amman. We had a short flight to ...,"{'intent': 'Flight route and connections', 'na...",Flight route and connections,"Tel Aviv to Amman, connecting to Heathrow and ...",Neutral,[]
1,0,Tel Aviv to Amman. We had a short flight to ...,"{'intent': 'In-flight smoking issue', 'named_e...",In-flight smoking issue,There was smoking on the plane,Negative,[]
2,0,Tel Aviv to Amman. We had a short flight to ...,"{'intent': 'Flight attendant service', 'named_...",Flight attendant service,The stewardess spent the entire time on her ce...,Negative,[]
3,1,Flight got delayed when I was flying to DR f...,"{'intent': 'Flight delay', 'named_entities': [...",Flight delay,Flight got delayed for an hour,Negative,[Frontier]
4,1,Flight got delayed when I was flying to DR f...,"{'intent': 'Flight schedule change', 'named_en...",Flight schedule change,Flight got advanced by 30 minutes,Neutral,[Frontier]
